In [1]:
import sys
import time
import numpy as np
from pynq import Overlay, allocate
 
sys.path.insert(0, "/home/xilinx/pynq/overlays/cnn_accel")
from model_training.load_weights import WeightLoader, decode_output

BIT_FILE    = "cnn_accel.bit"
WEIGHT_DIR  = "model_training/weights"
DMA_NAME    = "axi_dma_0"
 
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]

In [2]:
ol  = Overlay(BIT_FILE)
dma = getattr(ol, DMA_NAME)

loader = WeightLoader(WEIGHT_DIR, dma, ol.kernel_0)
loader.preload()
 
recv_buf = allocate(shape=(5,), dtype=np.uint32)

Pre-loading weight files…
  layer1_conv.bin: 128 packets cached
  layer2_conv.bin: 2,944 packets cached
  layer3_conv.bin: 11,264 packets cached
  layer4_conv.bin: 44,544 packets cached
  layer5_mlp.bin: 39,424 packets cached
  layer6_mlp.bin: 5,120 packets cached
  layer7_mlp.bin: 110 packets cached


In [5]:
image = np.zeros((32, 32, 3), dtype=np.uint8)

t0    = time.time()
probs = loader.run_inference(image, recv_buf)
dt    = (time.time() - t0) * 1000

print(f"\nInference time: {dt:.1f} ms")
print(f"\nRaw output words: {[hex(int(recv_buf[i])) for i in range(5)]}")
print(f"\nProbabilities:")
for i, (cls, p) in enumerate(zip(CLASSES, probs)):
    bar = "█" * int(p * 40)
    print(f"  {i:2d}  {cls:<12}  {p:.4f}  {bar}")
print(f"\nPredicted class: {CLASSES[np.argmax(probs)]}")
print(f"Prob sum: {probs.sum():.4f}  (should be ~1.0)")

del recv_buf
ol.free()


Inference time: 7149.6 ms

Raw output words: ['0x1b1619ec', '0x1afa19de', '0x178a1957', '0x195716e8', '0x1afa1a07']

Probabilities:
   0  airplane      0.1058  ████
   1  automobile    0.1013  ████
   2  bird          0.1054  ████
   3  cat           0.1010  ████
   4  deer          0.0920  ███
   5  dog           0.0990  ███
   6  frog          0.0990  ███
   7  horse         0.0895  ███
   8  ship          0.1054  ████
   9  truck         0.1017  ████

Predicted class: airplane
Prob sum: 0.9999  (should be ~1.0)
